I will attempt to cover all stages of the pipeline from this notebook before building the actual cookie-cutter python scripts
The stages are:
- Acquiring input (pre-recorded for now)
- segmenting the input
- ingesting it to pre-trained OMNIASR_LLM_300M
- Showing output

# Check pre-requisites
Make sure `ffmpeg` is installed in your system and is available as 'Path' from your environment variables.
Install and configure it depending on what OS you use.
Also on your python environment (python virtual environments/conda), install omnilingual-asr by running the code below

In [ ]:
% pip install omnilingual-asr

In [9]:
#Install import pytorch
import torch
import torchaudio

print(torch.__version__)
print(torchaudio.__version__)

2.8.0+cu128
2.8.0+cu128


# Stage 1: Acquiring Input
All raw data must be passed to `data/` folder, which model is to read

In [8]:
import subprocess
from pathlib import Path

def encode_to_wav(audio):
    encoded_audio = subprocess.run(
        ['ffmpeg', '-i', audio, '-f', 'wav', 'pipe:1'],
        check = True,
        capture_output= True
    )
    return encoded_audio.stdout #stdout is the actual file

data_path = Path('../data/')
audio = Path(data_path, 'Courtroom With Sarah Sothenes.m4a') #Enter name of audio
encoded_audio = encode_to_wav(audio)
type(encoded_audio) #type(encoded_audio)

bytes

In [ ]:
#obtained bytes go to a memory like object
import io, soundfile as sf

audio = io.BytesIO(encoded_audio)
audio.seek(0)

waveform, sr = sf.read(audio)
waveform = torch.from_numpy(waveform).float() #We must convert the waveforms into torch tensors

# Stage 2: Audio Segmenting (15-Second Chunks)
We are not to load huge amounts of audio into memory, the program will crash. So we have to segment it into 15 seconds for our model to ingest

In [15]:
def segment_audio(waveform, sr, chunk_duration= 15, overlap= 0.2):
    """
    Args:
        waveform: audio data in torch.Tensor format
        sr: sample rate i.e, number of audio samples in a second
        chunk_duration: how long a chunk is, defaults to 15 as defined in this function
        overlap: overlap ratio (0-1), eg. o.5 overlap means 50% overlap

    Returns:
        List of tuples, containing the chunk data, it's start time and end time
    """
    chunk_size = int(sr * chunk_duration)
    hop_size = int(chunk_size * (1 - overlap)) #stride between chunks

    chunks = []
    start_idx = 0

    while start_idx < len(waveform):
        end_idx = min(start_idx + chunk_size, len(waveform)) #for the last chunk, it's usually not the full chunk size
        chunk = waveform[start_idx:end_idx]

        start_time = start_idx / sr
        end_time = end_idx / sr

        chunks.append((chunk, start_time, end_time))
        start_idx += hop_size

    return chunks

In [18]:
chunks = segment_audio(waveform, sr)
print(f"Total chunks: {len(chunks)}")

Total chunks: 250


# Stage 3: Transcripting
We are going to initialize the model so it transcribes for us

In [19]:
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
pipeline = ASRInferencePipeline(model_card= 'omniASR_LLM_300M')

Output()

In [20]:
target_langs = ['eng_Latn', 'swh_Latn']

In [21]:
import tempfile
transcripts = []
for chunk, start_time, end_time in chunks:
    with tempfile.NamedTemporaryFile(suffix= '.wav', delete= False) as tmp:
        sf.write(tmp.name, chunk.numpy(), sr)
        transcript = pipeline.transcribe([tmp.name], batch_size= 1, lang= ['swh_Latn'])[0]
        transcripts.append({
            'text': transcript,
            'start': start_time,
            'end': end_time,
        })
        print(f"{start_time:.1f}s - {end_time:.1f}s: {transcript}")

0.0s - 15.0s: wakamilia nakuu wa wenye hekima walipumbatika wakaukamia utukufu wa mungu asiye na walirudi kwa mfano na sura vile damu aliye na walirudi na jengeni na wanyama na vitu vitabu
12.0s - 27.0s: ligi na wanyama na vitu vitamangu basi thirty six hivyo mungu aliwaachi wafuate ta- tamaa za- zao zaidi hata wanawaachi wakaba
24.0s - 39.0s: dhaivu hata wanawake wakabarili ma- matumizi ya asili kwa matumizi ya si- ya sio ya asili wanaume iko hivyo waliacha matumizi ya mtu ya asili waka- wakawaki- kiana
36.0s - 51.0s: si waka- wakawaki- kiana tambaa wanaume wakiandilia sio basi wakapata nafsi wao walipe wako tengo wao waliona kiam okay uhuh uwarudi sura ya pili stembo
48.0s - 63.0s: uhuh uwalume sura ya kili standway tafuta task last west we walume step ya two west we we we tuna nami u- uwahukumie wale wafanyia hao haiya nakutembea
60.0s - 75.0s: kununui wale wafanyao haya na kutenda yao au mwenyewe hiyo wadhani ya kwamba nitajiokusha na hukumi hakuna ehhe teke okay kae sijawahi kae s

KeyboardInterrupt: 